# NIH RePORTER ETL — Tobacco, Nicotine & ENDS Research Funding (FY2020–2024)

## Pipeline overview
1. **Extract** — POST paginated queries to the [NIH RePORTER Project API v2](https://api.reporter.nih.gov/) (`/v2/projects/search`), batched by fiscal year.
2. **Transform** — Flatten nested JSON, type-cast amounts/years, dedupe on `appl_id` + `fiscal_year`, parse congressional districts.
3. **Load** — Write one analysis-ready CSV to `data/` (relative to this notebook).

## API source
- Endpoint: `POST https://api.reporter.nih.gov/v2/projects/search`
- Criteria: RCDC spending categories **852 (Tobacco), 791 (Smoking and Health), 4721 (Electronic Nicotine Delivery Systems)** with `match_all: false` (OR logic); fiscal years 2020–2024; US organizations only; subprojects excluded.

In [ ]:
import re
import time
from pathlib import Path

import pandas as pd
import requests

# --- Config ---
API_URL = "https://api.reporter.nih.gov/v2/projects/search"
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)
RATE_LIMIT_SEC = 1.1  # NIH recommends <= 1 request/sec
FY_2020_2024 = [2020, 2021, 2022, 2023, 2024]
US_ONLY = {"org_countries": ["UNITED STATES"], "exclude_subprojects": True}


def post_search(payload):
    """POST to NIH RePORTER v2 with basic error surfacing."""
    resp = requests.post(API_URL, json=payload, timeout=90)
    resp.raise_for_status()
    return resp.json()


def fetch_page(criteria, offset=0, limit=500, include_fields=None):
    payload = {
        "criteria": criteria,
        "offset": offset,
        "limit": limit,
        # Sort on a unique key: batches are single-FY, so sorting by fiscal_year
        # gives unstable page ordering (dupes/skips across pages).
        "sort_field": "appl_id",
        "sort_order": "asc",
    }
    if include_fields:
        payload["include_fields"] = include_fields
    return post_search(payload)


def fetch_all_projects(criteria, include_fields=None, max_records=15000):
    """Paginate project search. Offset max is 14,999; limit max is 500."""
    all_results = []
    offset = 0
    limit = 500
    meta = {}
    while offset < min(max_records, 15000):
        data = fetch_page(criteria, offset=offset, limit=limit, include_fields=include_fields)
        meta = data.get("meta", {})
        batch = data.get("results") or []
        all_results.extend(batch)
        total = meta.get("total", 0)
        if not batch or offset + len(batch) >= total or len(all_results) >= max_records:
            break
        offset += limit
        time.sleep(RATE_LIMIT_SEC)
    return all_results, meta


def fetch_by_fiscal_years(criteria, fiscal_years, include_fields=None):
    """Batch API calls by fiscal year (rate-limit courtesy + offset safety)."""
    all_results = []
    totals = []
    for fy in fiscal_years:
        fy_criteria = {**criteria, "fiscal_years": [fy]}
        records, meta = fetch_all_projects(fy_criteria, include_fields=include_fields)
        all_results.extend(records)
        totals.append(meta.get("total", len(records)))
        time.sleep(RATE_LIMIT_SEC)
    return all_results, {"total": sum(totals), "fiscal_years": fiscal_years}


def _first_or_join(items, key="abbreviation", sep="; "):
    if not items:
        return None
    if isinstance(items, list):
        if key is None:
            vals = [str(i) for i in items if i is not None]
        else:
            vals = []
            for i in items:
                if isinstance(i, dict):
                    v = i.get(key)
                else:
                    v = i
                if v is not None and v != "":
                    vals.append(str(v))
        return sep.join(vals) if vals else None
    return str(items) if items is not None else None


def flatten_project(rec):
    """Flatten one API record to a flat row."""
    org = rec.get("organization") or {}
    admin = rec.get("agency_ic_admin") or {}
    fundings = rec.get("agency_ic_fundings") or []
    cong_dist = org.get("org_cong_dist") or rec.get("cong_dist")
    org_state = org.get("org_state")
    if not org_state and cong_dist:
        m = re.match(r"^([A-Z]{2})-", str(cong_dist).strip())
        org_state = m.group(1) if m else None
    org_type = org.get("org_type") or rec.get("organization_type")
    if isinstance(org_type, dict):
        org_type = org_type.get("name")
    return {
        "appl_id": rec.get("appl_id"),
        "fiscal_year": rec.get("fiscal_year"),
        "core_project_num": rec.get("core_project_num"),
        "project_title": rec.get("project_title"),
        "award_amount": rec.get("award_amount"),
        "activity_code": rec.get("activity_code"),
        "org_city": org.get("org_city"),
        "org_state": org_state,
        "org_country": org.get("org_country"),
        "cong_dist": cong_dist,
        "organization_type": org_type,
        "admin_ic": admin.get("abbreviation"),
        "admin_ic_name": admin.get("name"),
        "funding_ics": _first_or_join(fundings, key="abbreviation"),
        "spending_categories": _first_or_join(rec.get("spending_categories") or [], key="name", sep=" | "),
        "spending_categories_desc": _first_or_join(rec.get("spending_categories_desc") or [], key=None, sep=" | "),
        "funding_mechanism": (rec.get("funding_mechanism") or {}).get("name") if isinstance(rec.get("funding_mechanism"), dict) else rec.get("funding_mechanism"),
        "covid_response": _first_or_join(rec.get("covid_response") or [], key=None),
    }


def records_to_df(records):
    if not records:
        return pd.DataFrame()
    return pd.DataFrame([flatten_project(r) for r in records])


def standard_transform(df, dedupe_cols=None):
    """Common type-cast, dedupe, null-award logging."""
    if df.empty:
        return df
    out = df.copy()
    out["award_amount"] = pd.to_numeric(out["award_amount"], errors="coerce")
    out["fiscal_year"] = pd.to_numeric(out["fiscal_year"], errors="coerce").astype("Int64")
    null_amt = out["award_amount"].isna().sum()
    if null_amt:
        print(f"  null award_amount rows: {null_amt}")
    dedupe_cols = dedupe_cols or ["appl_id", "fiscal_year"]
    out = out.drop_duplicates(subset=dedupe_cols)
    return out


def basic_qa(df, label):
    print(f"--- QA: {label} ---")
    print(f"rows: {len(df):,} | cols: {df.shape[1]}")
    if df.empty:
        print("(empty dataframe)")
        return
    key_cols = [c for c in ["appl_id", "fiscal_year", "award_amount", "org_state", "admin_ic"] if c in df.columns]
    print("null share (key cols):")
    print((df[key_cols].isna().mean().round(3) * 100).astype(str) + "%")
    if "appl_id" in df.columns and "fiscal_year" in df.columns:
        dupes = df.duplicated(subset=["appl_id", "fiscal_year"]).sum()
        print(f"duplicate appl_id+fiscal_year: {dupes}")


def save_topic_csv(df, slug):
    path = DATA_DIR / f"{slug}.csv"
    df.to_csv(path, index=False)
    return path


def run_topic(slug, criteria, transform_fn, fiscal_years=None, include_fields=None):
    """Fetch (batched by FY) -> clean -> save -> return df."""
    print(f"Fetching {slug}...")
    fy_list = fiscal_years or criteria.get("fiscal_years", [])
    if fy_list:
        records, meta = fetch_by_fiscal_years(criteria, fy_list, include_fields=include_fields)
    else:
        records, meta = fetch_all_projects(criteria, include_fields=include_fields)
    print(f"  API total: {meta.get('total', '?')} | retrieved: {len(records)}")
    df_raw = records_to_df(records)
    df = transform_fn(df_raw)
    basic_qa(df, slug)
    path = save_topic_csv(df, slug)
    print(f"  saved -> {path}")
    return df

## Tobacco, Nicotine & ENDS Research Funding Explorer

**Exploration question:** How is NIH tobacco/nicotine/ENDS research funding distributed across states, institutes, and fiscal years?

**API criteria:** `spending_categories` [852, 791, 4721] (`match_all: false`); FY2020–2024; US orgs; exclude subprojects

**Output:** `data/tobacco_nih_awards.csv`

In [ ]:
tobacco_criteria = {
    **US_ONLY,
    "spending_categories": {"values": [852, 791, 4721], "match_all": False},
}

def transform_tobacco(df):
    out = standard_transform(df)
    if out.empty:
        return out
    dist = out["cong_dist"].str.extract(r"^([A-Z]{2})-(\d+)")
    out["cong_dist_state"] = dist[0]
    out["cong_dist_num"] = pd.to_numeric(dist[1]).astype("Int64")
    return out.sort_values(["fiscal_year", "award_amount"], ascending=[True, False])

df_tobacco = run_topic(
    "tobacco_nih_awards", tobacco_criteria, transform_tobacco,
    fiscal_years=FY_2020_2024,
    include_fields=[
        "ApplId", "FiscalYear", "AwardAmount", "ActivityCode", "ProjectTitle",
        "Organization", "CongDist", "AgencyIcAdmin", "AgencyIcFundings",
        "FundingMechanism", "OrganizationType", "SpendingCategories",
        "SpendingCategoriesDesc", "CoreProjectNum",
    ],
)

print(df_tobacco.shape)
print(df_tobacco.groupby("fiscal_year")["award_amount"].agg(["count", "sum"]))

## Export Summary


In [ ]:
print(f"{len(df_tobacco):,} award records -> data/tobacco_nih_awards.csv")